In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

print("🔄 Lendo a tabela bruta de ocupação e diária média do Unity Catalog...")
# Lê os dados originais carregados no catálogo
df_bruto = spark.table("main.default.ocup_e_dm")

print("🧼 Padronizando e renomeando as colunas para o formato Delta (snake_case)...")
# Dicionário de mapeamento para tratar caracteres especiais (%, aspas, acentos e espaços)
mapeamento_colunas = {
    "Data": "data",
    "%": "taxa_ocupacao",
    "PAXs": "paxs",
    "in's": "check-ins",
    "out's": "check-outs",
    "Diária Média": "diaria_media"
}

# Aplica o mapeamento de forma dinâmica no DataFrame
df_final = df_bruto
for col_original, col_nova in mapeamento_colunas.items():
    if col_original in df_final.columns:
        df_final = df_final.withColumnRenamed(col_original, col_nova)

# 3. Gravação na camada Bronze do Delta Lake
print("💾 Salvando dados tratados na tabela bronze.ocupacao_diaria_raw...")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.ocupacao_diaria_raw")

print("✅ Pipeline executado com sucesso! Tabela bronze.ocupacao_diaria_raw criada e protegida.")

In [0]:
%sql
SELECT * FROM bronze.ocupacao_diaria_raw LIMIT 10;